In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
con = sqlite3.connect("../vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "../v33.db" AS v33')
cur.execute('ATTACH DATABASE "../verb_patterns_isikud.db" AS isikud')

In [3]:
query = """
SELECT * 
from 
patterns_isikud_len1_alati
limit 20
"""

source2 = pd.read_sql_query(query, con)
source2

,ID,word,government,verb_word,compound_prt1,compound_prt2,compound_prt3,w_case,adp,verb,other,deprel,phrase_nr
0,1,küsima,kellelt/millelt,küsima,,,,abl,,,,obl,1
1,2,nõudma,kellelt/millelt,nõudma,,,,abl,,,,obl,1
2,3,ootama,kellelt/millelt,ootama,,,,abl,,,,obl,1
3,4,ostma,kellelt/millelt,ostma,,,,abl,,,,obl,1
4,5,paluma,kellelt/millelt,paluma,,,,abl,,,,obl,1
5,6,kuulma,kellelt/millelt,kuulma,,,,abl,,,,obl,1
6,7,tellima,kellelt/millelt,tellima,,,,abl,,,,obl,1
7,8,võtma ära,kellelt/millelt,võtma,ära,,,abl,,,,obl,1
8,9,tahtma,kellelt/millelt,tahtma,,,,abl,,,,obl,1
9,10,laekuma,kellelt/millelt,laekuma,,,,abl,,,,obl,1


#### transactions tabelist sõnad, mis on seotud annotatsiooniga 

- alati isikumäärus -> isik
- mitte kunagi isikumäärus -> mitte isik (eeldatavalt koht/sündmus)



In [3]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_wcomps_obl
""")

cur.execute("""
Create table transactions_verbs_wcomps_obl as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb,
    tbl1.verb_compound as verb_compound,
    tr.id as transaction_id,
    tr.lemma as root_word,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus

FROM 

transaction_head as tbl1

join 

transaction_v2 as tr

on 
    tbl1.id = tr.head_id
where
tr.deprel = 'obl'
""")


CPU times: user 30.2 s, sys: 8.06 s, total: 38.3 s
Wall time: 45.6 s


### võtta "alati" isikud verbidega seotud sõnad

In [33]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_wcomps_obl_alati
""")

cur.execute("""
Create table transactions_verbs_wcomps_obl_alati as
SELECT distinct
verbs.verb, verbs.verb_compound, verbs.root_word, verbs.koht, verbs.elus,tbl1.w_case, 'alati' as isik
from 
transactions_verbs_wcomps_obl as verbs
join 
isikud.patterns_isikud_len1_alati as tbl1
on 
tbl1.verb_word = verbs.verb
and tbl1.compound_prt1 = verbs.verb_compound
and INSTR(',' || verbs.tr_feats || ',', ',' || tbl1.w_case || ',') > 0
""")

CPU times: user 11 s, sys: 536 ms, total: 11.6 s
Wall time: 11.7 s


### võtta "mitte kunagi" isikud verbidega seotud sõnad

In [34]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_wcomps_obl_mitte_kunagi
""")

cur.execute("""
Create table transactions_verbs_wcomps_obl_mitte_kunagi as
SELECT distinct
verbs.verb, verbs.verb_compound, verbs.root_word, verbs.koht, verbs.elus,tbl1.w_case, 'mitte kunagi' as isik
from 
transactions_verbs_wcomps_obl as verbs
join 
isikud.patterns_isikud_len1_mitte_kunagi as tbl1
on 
tbl1.verb_word = verbs.verb
and tbl1.compound_prt1 = verbs.verb_compound
and INSTR(',' || verbs.tr_feats || ',', ',' || tbl1.w_case || ',') > 0
""")

CPU times: user 46.2 s, sys: 10.7 s, total: 56.9 s
Wall time: 57.4 s


## transactions obl tabel, kus on juures isik = alati/mitte kunagi

In [39]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_wcomps_obl_isik_tmp
""")

cur.execute("""
Create table transactions_verbs_wcomps_obl_isik_tmp as
SELECT verbs.*,tbl1.w_case, tbl1.isik
from 
transactions_verbs_wcomps_obl as verbs
left join
transactions_verbs_wcomps_obl_alati as tbl1
on 
verbs.verb = tbl1.verb
and verbs.verb_compound = tbl1.verb_compound
and verbs.root_word = tbl1.root_word
and INSTR(',' || verbs.tr_feats || ',', ',' || tbl1.w_case || ',') > 0

""")

CPU times: user 33.8 s, sys: 8.71 s, total: 42.5 s
Wall time: 45.1 s


In [41]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_wcomps_obl_isik
""")

cur.execute("""
Create table transactions_verbs_wcomps_obl_isik as
SELECT verbs.*,tbl1.w_case as w_case2, tbl1.isik as isik2
from 
transactions_verbs_wcomps_obl_isik_tmp as verbs
left join
transactions_verbs_wcomps_obl_mitte_kunagi as tbl1
on 
verbs.verb = tbl1.verb
and verbs.verb_compound = tbl1.verb_compound
and verbs.root_word = tbl1.root_word
and INSTR(',' || verbs.tr_feats || ',', ',' || tbl1.w_case || ',') > 0

""")

CPU times: user 1min 1s, sys: 34.1 s, total: 1min 35s
Wall time: 1min 48s


In [ ]:
cur.execute("""
ALTER TABLE transactions_verbs_wcomps_obl_isik
ADD isikumaarus text NOT NULL DEFAULt('') 
""")
con.commit()

In [44]:
cur.execute("""
UPDATE transactions_verbs_wcomps_obl_isik
SET isik = 'mitte kunagi'
WHERE isik2 = 'mitte kunagi'
""")
con.commit()

In [46]:
cur.execute("""
UPDATE transactions_verbs_wcomps_obl_isik
SET w_case = w_case2
WHERE w_case2 is not null
""")
con.commit()

In [50]:
query = """
SELECT 
*
from 
transactions_verbs_wcomps_obl_isik
limit 10
"""

source2 = pd.read_sql_query(query, con)
source2

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
0,2,toimuma,,1,lõpp,obl,S,"com,in,sg",UNK,UNK,in,mitte kunagi,in,mitte kunagi
1,2,toimuma,,3,1.,obl,N,"<?>,ord,roman",UNK,UNK,None,None,None,None
2,3,saama,pihta,7,keel,obl,S,"all,com,pl",UNK,UNK,None,None,None,None
3,6,kulmineeruma,,15,purukspeksmine,obl,S,"com,kom,sg",UNK,UNK,None,None,None,None
4,10,tulema,,19,sina,obl,P,"ad,sg",UNK,YES,ad,alati,None,None
5,11,viilima,,22,tund,obl,S,"com,el,pl",UNK,UNK,None,None,None,None
6,11,viilima,,23,juht,obl,S,"ad,com,sg",UNK,YES,None,None,None,None
7,16,tulema,,25,mis,obl,P,"gen,pl",UNK,UNK,None,None,None,None
8,23,alustama,,36,muusika,obl,S,"com,kom,sg",UNK,UNK,None,None,None,None
9,25,muutuma,,40,mis,obl,P,"el,sg",UNK,UNK,None,None,None,None


#### kontroll, et kõik obl transactionid on kaetud 

In [62]:
query = """
SELECT 
count(*)
from 
transactions_verbs_wcomps_obl_isik
"""

source2 = pd.read_sql_query(query, con)
source2

,count(*)
0,12714382


In [79]:
query = """
SELECT 
count(*)
from 
transaction_v2
where deprel = 'obl'
"""

source2 = pd.read_sql_query(query, con)
source2

,count(*)
0,12714382


## Create necessary tables for plotting

In [64]:
# base table with root counts

cur.execute("""
drop table if exists transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_base
""")

cur.execute("""
create table transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_base as
SELECT root_word, count(root_word) as root_cnt
from 
transactions_verbs_wcomps_obl_isik
group by root_word
""")

In [67]:
# table counts elus 

cur.execute("""
drop table if exists transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_elus
""")

cur.execute("""
Create table transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_elus as
SELECT root_word, count(root_word) as elus_cnt
FROM 
transactions_verbs_wcomps_obl_isik
where isik = 'alati'
group by root_word
""")

In [68]:
# tbl count koht

cur.execute("""
drop table if exists transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_koht
""")

cur.execute("""
Create table transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_koht as
SELECT root_word, count(root_word) as mitte_elus_cnt
FROM 
transactions_verbs_wcomps_obl_isik
where isik = 'mitte kunagi'
group by root_word
""")

In [70]:
# join everything into 1 table

cur.execute("""DROP table if exists transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_1""")
cur.execute("""
CREATE TABLE transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_1 AS
select tbl1.root_word, tbl1.root_cnt, elus_cnt
from
transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_base as tbl1
left join
transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_elus as tbl2
on tbl1.root_word=tbl2.root_word 
""")


cur.execute("""DROP table if exists transactions_verbs_wcomps_obl_isik_root_eluskoht_counts""")
cur.execute("""
CREATE TABLE transactions_verbs_wcomps_obl_isik_root_eluskoht_counts AS
select tbl1.root_word, tbl1.root_cnt, tbl1.elus_cnt, mitte_elus_cnt
from
transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_1 as tbl1
left join
transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_koht as tbl2
on tbl1.root_word=tbl2.root_word 
""")

In [71]:
# update table null -> 0

cur.execute("""
UPDATE transactions_verbs_wcomps_obl_isik_root_eluskoht_counts
SET elus_cnt = 0
where elus_cnt is null
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_wcomps_obl_isik_root_eluskoht_counts
SET mitte_elus_cnt = 0
where mitte_elus_cnt is null
""")
con.commit()

In [74]:
query = """
SELECT * 
from  transactions_verbs_wcomps_obl_isik_root_eluskoht_counts
where elus_cnt!=mitte_elus_cnt
limit 20
"""

source2 = pd.read_sql_query(query, con)
source2

,root_word,root_cnt,elus_cnt,mitte_elus_cnt
0,$,321,0,1
1,$1,2,0,1
2,%,32,0,4
3,%-ilis,1,0,1
4,%-põhimõte,1,0,1
5,%-see,3,0,2
6,%line,54,8,34
7,-30s,1,0,1
8,-4%,17,0,2
9,-4.,3,0,1


In [75]:
query = """
SELECT * 
from 
transactions_verbs_wcomps_obl_isik_root_eluskoht_counts
where elus_cnt!=root_cnt and elus_cnt>0
limit 20
"""

source2 = pd.read_sql_query(query, con)
source2

,root_word,root_cnt,elus_cnt,mitte_elus_cnt
0,%line,54,8,34
1,000,309,1,14
2,007,42,1,1
3,1.,611,3,15
4,1.70,11,1,0
5,10,1099,1,8
6,10%,500,1,19
7,10.,246,1,20
8,100.,100,2,1
9,11-12aastane,3,1,0


In [82]:
query = """
SELECT * from 
transactions_verbs_wcomps_obl_isik_root_eluskoht_counts
"""

s = pd.read_sql_query(query, con)
s.to_csv("transactions_verbs_wcomps_obl_isik_root_eluskoht_counts.csv", sep=",", index=False, encoding="utf-8")

In [4]:
con.close()